### Notebook 范围

### 目的
检验 Z300 source metric 与 EPFz 在垂直方向上的相关是否连续。

### 科学问题
Z300 与 EP100 的关系是否只是 100 hPa 偶然相关，还是从 300/200 hPa 到 100/70/50 hPa 有垂直传播一致性？

### 方法
EPFz = mean(-ep2)，40-80N，pressure levels = 300, 200, 150, 100, 70, 50 hPa。source metric 使用 05 中 Jan/Feb 的 Z300 projection 和 wave2 amplitude。

### 输出
outputs/figures/06_vertical_chain 和 outputs/tables/06_vertical_chain。

### 解读
如果相关在 300-100 hPa 连续，支持 tropospheric source -> lower stratosphere wave propagation 的链条。

### 注意
相关不证明因果；EP flux 是 zonal-mean wave-mean-flow diagnostic，无经度维。


### 导入与公共路径

### 目的
加载 Extention_analysis 公共函数。

### 科学问题
所有 notebook 是否共享相同路径、变量定义和 sign convention？

### 方法
导入 hindcast_extension_utils.py。

### 输出
无图。

### 解读
路径正确即可继续。

### 注意
所有输出都写入 outputs 子目录。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 绘制 vertical propagation profile

### 目的
计算 Z300 projection/wave2 amplitude 与各层 EPFz W1+W2/wave2 的相关。

### 科学问题
source metric 的影响是否从对流层上传到 100 hPa？

### 方法
优先使用 05 的 Jan/Feb metrics；每个 case-month 与 pressure level 计算 Pearson R。

### 输出
06_Z300_to_EPFz_vertical_profiles_JanFeb_v2.png/pdf 和 CSV。

### 解读
随高度连续的高相关比单层 100 hPa 相关更像物理传播链条。

### 注意
只有 Jan/Feb 可用窗口参与；later init 不强行解释 January source。


In [ ]:
fig_dir = figure_dir("06_vertical_chain")
tab_dir = table_dir("06_vertical_chain")
metrics_path = TAB_DIR / "05_Z300_source_metrics_all_cases.csv"
metrics = pd.read_csv(metrics_path, dtype={"member": str, "case": str, "year": str}) if metrics_path.exists() else pd.DataFrame()
if not metrics.empty:
    metrics["member"] = metrics["member"].astype(str).str.zfill(3)
levels = [300, 200, 150, 100, 70, 50]
rows = []
for (case, month), sub in metrics.groupby(["case", "month"]) if not metrics.empty else []:
    window = MONTH_WINDOWS[month]
    for wave, target_metric, zmetric in [
        ("wave1", "EPFz_wave1", "Z300_stationary_projection_to_B2000"),
        ("wave2", "EPFz_wave2", "Z300_wave2_amplitude_m"),
        ("all_waves", "EPFz_all", "Z300_stationary_projection_to_B2000"),
    ]:
        prof = compute_ep_vertical_profile(case, wave, window, levels_hpa=levels)
        if not prof.empty:
            prof["member"] = prof["member"].astype(str).str.zfill(3)
        if prof.empty:
            continue
        merged = sub[["member", zmetric]].merge(prof, on="member", how="inner")
        for lev, levsub in merged.groupby("plev_hpa"):
            c = finite_corr(levsub[zmetric], levsub["EPFz"])
            rows.append({"case": case, "month": month, "wave": wave, "z_metric": zmetric, "plev_hpa": lev, **c})
vert = pd.DataFrame(rows)
vert_csv = tab_dir / "06_Z300_to_EPFz_vertical_correlations_JanFeb.csv"
vert.to_csv(vert_csv, index=False)
vert.to_csv(TAB_DIR / "06_Z300_to_EPFz_vertical_correlations_all_cases.csv", index=False)
selected = vert.dropna(subset=["R"]).groupby(["case", "month"]).size().reset_index()[['case','month']].head(6).itertuples(index=False, name=None)
selected = list(selected)
fig, axes = plt.subplots(1, max(1, len(selected)), figsize=(4.0 * max(1, len(selected)), 5.2), sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
if not selected:
    axes[0].axis("off"); axes[0].text(0.5, 0.5, "No vertical correlations", ha="center")
else:
    for ax, (case, month) in zip(axes, selected):
        sub = vert[(vert["case"] == case) & (vert["month"] == month)]
        for wave, color in [("all_waves", "black"), ("wave1", "tab:blue"), ("wave2", "tab:orange")]:
            s = sub[sub["wave"] == wave].sort_values("plev_hpa", ascending=False)
            if not s.empty:
                ax.plot(s["R"], s["plev_hpa"], marker="o", label=wave, color=color)
        ax.axvline(0, color="0.3", lw=0.8)
        ax.set_xbound(-1, 1)
        ax.set_yscale("log"); ax.invert_yaxis(); ax.set_yticks(levels); ax.set_yticklabels([str(v) for v in levels])
        ax.set_title(f"{case} {month}")
        ax.set_xlabel("R")
    axes[0].set_ylabel("Pressure (hPa)")
    axes[-1].legend(fontsize=8)
savefig(fig, "06_Z300_to_EPFz_vertical_profiles_JanFeb_v2", fig_dir=fig_dir, notebook="06_vertical_propagation_chain.ipynb", scientific_question="Z300 source metric 与 EPFz 是否有垂直连续性？", variables_windows="Jan/Feb; EPFz=mean(-ep2), 40-80N, 300-50 hPa; Z300 projection/wave2 amplitude", interpretation="相关从 300/200 到 100 hPa 连续支持 vertical propagation chain。", caveat="相关不是因果证明；EP flux 是 zonal-mean diagnostic。", csv_table=vert_csv)
plt.show()
write_figure_guide()
